In [ ]:
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
import numpy as np
import pickle
import json
from pathlib import Path

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
def calculate_csd(sim_path):
    """Calculates cloud statistics for a given simulation."""

    # model grid parameters
    dx, dy, dz = 250, 250, 50 # m
    grid_vol = dx*1.0e-3*dy*1.0e-3*dz*1.0e-3 # km**3

    # tunable parameters
    # the highest cloud base level (above domain mean cloud base level)
    # allowed to be considered an attached cloud
    cbl_gap = 3 
    # cloud liquid water mixing ratio criterion
    # for determining domain mean cloud base
    qclm_crit = 1.0e-6 # g/kg

    sim_path = Path(sim_path)

    with open(sim_path / 'uninterrupted_large_clouds.json', 'r') as f:
        ul_clouds = json.load(f)
    ul_clouds_list = list(ul_clouds.keys())
    nc = len(ul_clouds_list)
    id_list = np.asarray(ul_clouds_list, dtype=np.int32)

    with open(sim_path / 'pkl/cloud_all_af.pkl', 'rb') as f:
        cloud_areas = pickle.load(f)
    with open(sim_path / 'pkl/plume_all_af.pkl', 'rb') as f:
        plume_areas = pickle.load(f)

    max_plume_area = plume_areas.max(axis=(1,2))
    max_cloud_area = cloud_areas.max(axis=(1,2))
    cloud_volumes = cloud_areas.sum(axis=2)
    cloud_times = (cloud_volumes > 0).sum(axis=1)
    cloud_ini_time = np.argmax(cloud_volumes > 0, axis=1)

    total_cloud_volumes = cloud_volumes.sum(axis=1) * grid_vol # in km**3   

    cbl_c = np.argmax(cloud_areas > 0, axis=2)
    # min cloud base and max cloud top for each cloud
    mcbl_c = np.zeros(nc, dtype=np.int32)
    mctl_c = np.zeros(nc, dtype=np.int32)
    for i in range(nc):
        a = cloud_areas[i,:,:].sum(axis=0)
        mcbl_c[i] = ((np.where(a != 0))[0]).min()
        mctl_c[i] = ((np.where(a != 0))[0]).max()

    with open(sim_path / 'pkl/qclm.pkl', 'rb') as f:
        qclm = np.asarray(pickle.load(f))
    mcbl = np.argmax(qclm > qclm_crit, axis=1)

    attached_c = (cloud_volumes > 0) & (cbl_c - cbl_gap < mcbl[np.newaxis, :] )
    attached = attached_c.sum(axis=1) > 0
    attached_clouds = id_list[attached]
    attached_ind = np.asarray(np.nonzero(np.isin(id_list, attached_clouds, assume_unique=True))[0])

    with open(sim_path / 'pkl/plume_all_mf.pkl', 'rb') as f:
        plume_mfs = pickle.load(f)
    plume_mfs = plume_mfs*dx*dy # now in kg/s unit

    def calc_mean_mf(plume_mfs, mcbl, attached_c):
        nc, nt, _ = plume_mfs.shape
        mf = np.zeros(nc, dtype=float)
        mf_t = np.zeros((nc, nt), dtype=float)
        for i in range(nc):
            a = attached_c[i,:]
            for t in range(nt):
                # average over 3 levels around cloud base
                mf_t[i,t] = plume_mfs[i, t, mcbl[t]-1:mcbl[t]+2].mean()
            if a.sum() != 0:
                mf[i] = (a * mf_t[i,:]).sum()/a.sum()
        return mf

    mean_cb_mf = calc_mean_mf(plume_mfs, mcbl, attached_c)
    mean_cb_mf_c = np.where(mean_cb_mf > 1.0, mean_cb_mf, 1.1)
    clipped_mf = mean_cb_mf_c[attached_ind]

    return clipped_mf, mean_cb_mf[attached_ind], total_cloud_volumes[attached_ind], cloud_times[attached_ind], mcbl_c[attached_ind], mctl_c[attached_ind], cloud_ini_time[attached_ind], max_cloud_area[attached_ind], max_plume_area[attached_ind], attached_ind

In [ ]:
l_calc_csd = False
if l_calc_csd:
    case_name = 'goamazon_2pulse.largedom.r20251008.rerun'
    stat_ctl = calculate_csd(f'{case_name}/')
    with open(f'{case_name}/pkl/csd_stats.pkl', 'wb') as f:
        pickle.dump(stat_ctl, f)
    case_name = 'goamazon_2pulse.largedom.ehe1.r20251030.rerun'
    stat_ehe1 = calculate_csd(f'{case_name}/')
    with open(f'{case_name}/pkl/csd_stats.pkl', 'wb') as f:
        pickle.dump(stat_ehe1, f)

In [ ]:
ctl = 'goamazon_2pulse.largedom.r20251008.rerun'
ehe1 = 'goamazon_2pulse.largedom.ehe1.r20251030.rerun'
with open(f'{ctl}/pkl/csd_stats.pkl', 'rb') as f:
    stat_ctl = pickle.load(f)
with open(f'{ehe1}/pkl/csd_stats.pkl', 'rb') as f:
    stat_ehe1 = pickle.load(f)

In [ ]:
nc_ehe1, = stat_ehe1[0].shape
nc_ctl, = stat_ctl[0].shape
print(f'{nc_ehe1} clouds in EHE1 simulation')
print(f'{nc_ctl} clouds in CTL simulation')

In [ ]:
dz = 50. # m
z = np.arange(dz/2, 5000., dz)
dts = 30 # seconds

In [ ]:
minmf = 0
print(np.log10(np.max(stat_ehe1[0])))
print(np.log10(np.max(stat_ctl[0])))
maxmf = np.max([np.max(stat_ehe1[0]), np.max(stat_ctl[0])]) + 10.0
# maxmf = np.max(stat_ehe1[0]) + 10.0
maxmf = np.log10(maxmf)
print(minmf, maxmf)
mf_bins = np.linspace(minmf, maxmf, 16)

In [ ]:
time_bins = np.arange(0, 241, 30)
print(time_bins)

In [ ]:
fig = plt.figure(figsize=(12,7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
n, time_bins, patches = ax.hist([stat_ehe1[-4], stat_ctl[-4]], bins=time_bins, log=True, color=['red', 'black'])
ax.legend(['EHE1', 'CTL'])
ax.set_ylim(1, 3.0e4)
ax.set_xlabel('cloud ini time')
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i in range(len(time_bins) - 1):
    ax = axes[i]
    
    # Filter clouds by initialization time bin
    mask_ehe1 = (stat_ehe1[-4] > time_bins[i]) & (stat_ehe1[-4] <= time_bins[i+1])
    mask_ctl = (stat_ctl[-4] > time_bins[i]) & (stat_ctl[-4] <= time_bins[i+1])
    
    # Plot histograms for clouds in this time bin
    n_bin, _, _ = ax.hist([np.log10(stat_ehe1[0][mask_ehe1]), np.log10(stat_ctl[0][mask_ctl])], 
                          bins=mf_bins, log=True, color=['red', 'black'], alpha=0.7)
    
    ax.set_xlabel(f'Mean cloud-base mass flux (kg/s)')
    ax.set_ylabel('Count')
    ax.set_title(f'Time bin: {time_bins[i]:.0f}-{time_bins[i+1]:.0f}')
    ax.legend(['EHE1', 'CTL'])
    ax.set_ylim(0.5, 1.0e4)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Calculate difference histograms for each time bin
fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for i in range(len(time_bins) - 1):
    ax = axes[i]
    
    # Filter clouds by initialization time bin
    mask_ehe1 = (stat_ehe1[-4] > time_bins[i]) & (stat_ehe1[-4] <= time_bins[i+1])
    mask_ctl = (stat_ctl[-4] > time_bins[i]) & (stat_ctl[-4] <= time_bins[i+1])
    
    # Calculate histograms for this time bin
    n_ehe1, _ = np.histogram(np.log10(stat_ehe1[0][mask_ehe1]), bins=mf_bins)
    n_ctl, _ = np.histogram(np.log10(stat_ctl[0][mask_ctl]), bins=mf_bins)
    
    # Calculate difference
    diff = n_ehe1 - n_ctl
    bin_centers = (mf_bins[:-1] + mf_bins[1:]) / 2
    
    # Plot difference bars
    ax.bar(bin_centers, np.where(diff > 0, diff, np.nan), 
           width=np.diff(mf_bins), color='red', alpha=0.7, label=r'EHE1 $>$ CTL')
    ax.bar(bin_centers, np.where(diff < 0, -diff, np.nan), 
           width=np.diff(mf_bins), color='blue', alpha=0.7, label=r'CTL $>$ EHE1')
     
    ax.axhline(0, color='black', linestyle='-', linewidth=1)
    ax.set_xlabel('Mean cloud-base mass flux (log10 kg/s)')
    ax.set_ylabel('Absolute Difference in Count')
    ax.set_yscale('log')
    ax.set_ylim([0.1, 1.0e3])
    ax.set_title(f'Time bin: {time_bins[i]:.0f}-{time_bins[i+1]:.0f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(12,7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
n, mf_bins, patches = ax.hist([np.log10(stat_ehe1[0]), np.log10(stat_ctl[0])], bins=mf_bins, log=True, color=['red', 'black'])
ax.axvline(np.median(np.log10(stat_ehe1[0])), color='red', linestyle='--')
ax.axvline(np.median(np.log10(stat_ctl[0])), color='black', linestyle='--')
ax.legend(['EHE1', 'CTL'])
ax.set_ylim(1, 3.0e4)
ax.set_xlabel(f'Mean cloud-base mass flux (kg/s)')
plt.show()

In [ ]:
fig = plt.figure(figsize=(12,7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
diff = n[0] - n[1]
print(diff)
bin_centers = (mf_bins[:-1] + mf_bins[1:]) / 2
ax.bar(bin_centers, np.where(diff>0.0, diff, np.nan), width=np.diff(mf_bins), color='red', label=r'EHE1 $>$ CTL')
ax.bar(bin_centers, np.where(diff>0.0, np.nan, -diff), width=np.diff(mf_bins), color='blue', label=r'CTL $>$ EHE1')
ax.axhline(0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Mean cloud-base mass flux (kg/s)')
ax.set_ylabel('Difference (EHE1 - CTL)')
ax.set_yscale('log')
ax.set_ylim(0.1, 1.0e4)
ax.set_title('Difference between EHE1 and CTL histograms')
ax.legend()
plt.show()

In [ ]:
mean_cb_mf = stat_ctl[1]
cloud_times = stat_ctl[3]
mctl_c = stat_ctl[5]
cloud_volume = stat_ctl[2]
max_cloud_area = stat_ctl[-3]
max_plume_area = stat_ctl[-2]
print(f"corrcoef(<M_b>, cloud lifetime) = {np.corrcoef(mean_cb_mf, cloud_times)[0,1]}")
print(f"corrcoef(<M_b>, cloud top height) = {np.corrcoef(mean_cb_mf, z[mctl_c])[0,1]}") 
print(f"corrcoef(<M_b>, cloud volume) = {np.corrcoef(mean_cb_mf, cloud_volume)[0,1]}")
print(f"corrcoef(<M_b>, max cloud area) = {np.corrcoef(mean_cb_mf, max_cloud_area)[0,1]}")
print(f"corrcoef(<M_b>, max plume area) = {np.corrcoef(mean_cb_mf, max_plume_area)[0,1]}")

In [ ]:
clipped_mf = stat_ctl[0]
cloud_times = stat_ctl[3]

nbins = len(mf_bins) - 1
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))

n, _ = np.histogram(np.log10(clipped_mf), bins=mf_bins)
sum_t, _ = np.histogram(
    np.log10(clipped_mf), bins=mf_bins, weights=cloud_times * dts
)
ave_t_ctl = np.asarray(sum_t) / np.asarray(n, dtype=float)
bcs = (mf_bins[1:] + mf_bins[:-1]) * 0.5

bin_ids = np.digitize(np.log10(clipped_mf), mf_bins) - 1

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
for i, c in enumerate(colors):
    ax.scatter(
        np.log10(clipped_mf[bin_ids == i]),
        np.log10(cloud_times[bin_ids == i] * dts),
        s=5,
        color=c,
    )
    ax.plot(bcs[i], np.log10(ave_t_ctl[i]), marker="s", markersize=20, color='k')
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Cloud Lifetime (secs, log-10-scale)")
ax.set_ylim(1, 4)
ax.set_title('CTL - large')
plt.show()

In [ ]:
mctl_c = stat_ctl[5]

sum_z_ctl, _ = np.histogram(
    np.log10(clipped_mf), bins=mf_bins, weights=z[mctl_c]
)
ave_z_ctl = np.asarray(sum_z_ctl) / np.asarray(n, dtype=float)

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))
for i, c in enumerate(colors):
    ax.scatter(np.log10(clipped_mf[bin_ids == i]), z[mctl_c[bin_ids == i]], s=5, color=c)
    ax.plot(bcs[i], ave_z_ctl[i], marker="s", markersize=20, color='k')
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Cloud Top Height (m)")
ax.set_ylim((500, 5000))
# ax.set_xlim((2, 5.5))
ax.set_title('CTL - large')
plt.show()

In [ ]:
max_cloud_area = stat_ctl[-3]
print(clipped_mf.shape, max_cloud_area.shape)
s, _ = np.histogram(
    np.log10(clipped_mf), bins=mf_bins, weights=max_cloud_area
)
ave_mca_ctl = np.asarray(s) / np.asarray(n, dtype=float)
bcs = (mf_bins[1:] + mf_bins[:-1]) * 0.5
bin_ids = np.digitize(np.log10(clipped_mf), mf_bins) - 1

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))
for i, c in enumerate(colors):
    ax.scatter(np.log10(clipped_mf[bin_ids == i]), max_cloud_area[bin_ids == i], s=5, color=c)
    ax.plot(bcs[i], ave_mca_ctl[i], marker="s", markersize=20, color='k')
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Max Cloud Area (gp)")
ax.set_ylim((1, 150))
ax.set_yscale('log')
# ax.set_xlim((2, 5.5))
ax.set_title('CTL - large')
plt.show()

In [ ]:
max_plume_area = stat_ctl[-2]
print(clipped_mf.shape, max_plume_area.shape)
s, _ = np.histogram(
    np.log10(clipped_mf), bins=mf_bins, weights=max_plume_area
)
ave_mpa_ctl = np.asarray(s) / np.asarray(n, dtype=float)
bcs = (mf_bins[1:] + mf_bins[:-1]) * 0.5
bin_ids = np.digitize(np.log10(clipped_mf), mf_bins) - 1

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))
for i, c in enumerate(colors):
    ax.scatter(np.log10(clipped_mf[bin_ids == i]), max_plume_area[bin_ids == i], s=5, color=c)
    ax.plot(bcs[i], ave_mpa_ctl[i], marker="s", markersize=20, color='k')
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Max Plume Area (gp)")
ax.set_ylim((1, 300))
ax.set_yscale('log')
# ax.set_xlim((2, 5.5))
ax.set_title('CTL - large')
plt.show()

In [ ]:
cv = stat_ctl[2]
print(clipped_mf.shape, cv.shape)
s, _ = np.histogram(
    np.log10(clipped_mf), bins=mf_bins, weights=cv
)
ave_cv_ctl = np.asarray(s) / np.asarray(n, dtype=float)
bcs = (mf_bins[1:] + mf_bins[:-1]) * 0.5
bin_ids = np.digitize(np.log10(clipped_mf), mf_bins) - 1

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))
for i, c in enumerate(colors):
    ax.scatter(np.log10(clipped_mf[bin_ids == i]), cv[bin_ids == i], s=5, color=c)
    ax.plot(bcs[i], ave_cv_ctl[i], marker="s", markersize=20, color='k')
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Cloud Volume (km^3)")
ax.set_ylim((0.01, 1000))
ax.set_yscale('log')
# ax.set_xlim((2, 5.5))
ax.set_title('CTL - large')
plt.show()

In [ ]:
mean_cb_mf = stat_ehe1[1]
cloud_times = stat_ehe1[3]
mctl_c = stat_ehe1[5]
cloud_volume = stat_ehe1[2]
max_cloud_area = stat_ehe1[-3]
max_plume_area = stat_ehe1[-2]
print(f"corrcoef(<M_b>, cloud lifetime) = {np.corrcoef(mean_cb_mf, cloud_times)[0,1]}")
print(f"corrcoef(<M_b>, cloud top height) = {np.corrcoef(mean_cb_mf, z[mctl_c])[0,1]}") 
print(f"corrcoef(<M_b>, cloud volume) = {np.corrcoef(mean_cb_mf, cloud_volume)[0,1]}")
print(f"corrcoef(<M_b>, max cloud area) = {np.corrcoef(mean_cb_mf, max_cloud_area)[0,1]}")
print(f"corrcoef(<M_b>, max plume area) = {np.corrcoef(mean_cb_mf, max_plume_area)[0,1]}")

In [ ]:
clipped_mf = stat_ehe1[0]
cloud_times = stat_ehe1[3]

nbins = len(mf_bins) - 1
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))

n, _ = np.histogram(np.log10(clipped_mf), bins=mf_bins)
sum_t, _ = np.histogram(
    np.log10(clipped_mf), bins=mf_bins, weights=cloud_times * dts
)
ave_t_ehe1 = np.asarray(sum_t) / np.asarray(n, dtype=float)
bcs = (mf_bins[1:] + mf_bins[:-1]) * 0.5
bin_ids = np.digitize(np.log10(clipped_mf), mf_bins) - 1

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
for i, c in enumerate(colors):
    ax.scatter(
        np.log10(clipped_mf[bin_ids == i]),
        np.log10(cloud_times[bin_ids == i] * dts),
        s=5,
        color=c,
    )
    ax.plot(bcs[i], np.log10(ave_t_ehe1[i]), marker="s", markersize=20, color='k')
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Cloud Lifetime (min, log-10-scale)")
ax.set_title('EHE1 - large')
ax.set_ylim(1, 4)
plt.show()


In [ ]:
mctl_c = stat_ehe1[5]

sum_z, _ = np.histogram(
    np.log10(clipped_mf), bins=mf_bins, weights=z[mctl_c]
)
ave_z_ehe1 = np.asarray(sum_z) / np.asarray(n, dtype=float)

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))
for i, c in enumerate(colors):
    ax.scatter(np.log10(clipped_mf[bin_ids == i]), z[mctl_c[bin_ids == i]], s=5, color=c)
    ax.plot(bcs[i], ave_z_ehe1[i], marker="s", markersize=20, color='k')
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Cloud Top Height (m)")
ax.set_ylim((500, 5000))
ax.set_title('EHE1 - large')
# ax.set_xlim((2, 5.5))
plt.show()

In [ ]:
max_cloud_area = stat_ehe1[-3]
print(clipped_mf.shape, max_cloud_area.shape)
s, _ = np.histogram(
    np.log10(clipped_mf), bins=mf_bins, weights=max_cloud_area
)
ave_mca_ehe1 = np.asarray(s) / np.asarray(n, dtype=float)
bcs = (mf_bins[1:] + mf_bins[:-1]) * 0.5
bin_ids = np.digitize(np.log10(clipped_mf), mf_bins) - 1

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))
for i, c in enumerate(colors):
    ax.scatter(np.log10(clipped_mf[bin_ids == i]), max_cloud_area[bin_ids == i], s=5, color=c)
    ax.plot(bcs[i], ave_mca_ehe1[i], marker="s", markersize=20, color='k')
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Max Cloud Area (gp)")
ax.set_ylim((1, 150))
ax.set_yscale('log')
# ax.set_xlim((2, 5.5))
ax.set_title('EHE1 - large')
plt.show()

In [ ]:
max_plume_area = stat_ehe1[-2]
print(clipped_mf.shape, max_plume_area.shape)
s, _ = np.histogram(
    np.log10(clipped_mf), bins=mf_bins, weights=max_plume_area
)
ave_mpa_ehe1 = np.asarray(s) / np.asarray(n, dtype=float)
bcs = (mf_bins[1:] + mf_bins[:-1]) * 0.5
bin_ids = np.digitize(np.log10(clipped_mf), mf_bins) - 1

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))
for i, c in enumerate(colors):
    ax.scatter(np.log10(clipped_mf[bin_ids == i]), max_plume_area[bin_ids == i], s=5, color=c)
    ax.plot(bcs[i], ave_mpa_ehe1[i], marker="s", markersize=20, color='k')
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Max Plume Area (gp)")
ax.set_ylim((1, 300))
ax.set_yscale('log')
# ax.set_xlim((2, 5.5))
ax.set_title('EHE1 - large')
plt.show()

In [ ]:
cv = stat_ehe1[2]
print(clipped_mf.shape, cv.shape)
s, _ = np.histogram(
    np.log10(clipped_mf), bins=mf_bins, weights=cv
)
ave_cv_ehe1 = np.asarray(s) / np.asarray(n, dtype=float)
bcs = (mf_bins[1:] + mf_bins[:-1]) * 0.5
bin_ids = np.digitize(np.log10(clipped_mf), mf_bins) - 1

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))
for i, c in enumerate(colors):
    ax.scatter(np.log10(clipped_mf[bin_ids == i]), cv[bin_ids == i], s=5, color=c)
    ax.plot(bcs[i], ave_cv_ehe1[i], marker="s", markersize=20, color='k')
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Cloud Volume (km^3)")
ax.set_ylim((0.01, 1000))
ax.set_yscale('log')
# ax.set_xlim((2, 5.5))
ax.set_title('EHE1 - large')
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
ax.plot(bcs, ave_t_ctl, marker='o', markersize=10, label='CTL - large', color='black', linewidth=2)
ax.plot(bcs, ave_t_ehe1, marker='s', markersize=10, label='EHE1 - large', color='red', linewidth=2)
ax.set_xlabel(r'$\log_{10} \langle M_b \rangle$', fontsize=20)
ax.set_ylabel('Average Cloud Lifetime (s)', fontsize=20)
ax.legend(fontsize=16)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
ax.plot(bcs, ave_z_ctl, marker='o', markersize=10, label='CTL - large', color='black', linewidth=2)
ax.plot(bcs, ave_z_ehe1, marker='s', markersize=10, label='EHE1 - large', color='red', linewidth=2)
ax.set_xlabel(r'$\log_{10} \langle M_b \rangle$', fontsize=20)
ax.set_ylabel('Average Cloud Top Height (m)', fontsize=20)
ax.legend(fontsize=16)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
ax.plot(bcs, ave_mca_ctl, marker='o', markersize=10, label='CTL - large', color='black', linewidth=2)
ax.plot(bcs, ave_mca_ehe1, marker='s', markersize=10, label='EHE1 - large', color='red', linewidth=2)
ax.set_xlabel(r'$\log_{10} \langle M_b \rangle$', fontsize=20)
ax.set_ylabel('max cld area', fontsize=20)
ax.legend(fontsize=16)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
ax.plot(bcs, ave_mpa_ctl, marker='o', markersize=10, label='CTL - large', color='black', linewidth=2)
ax.plot(bcs, ave_mpa_ehe1, marker='s', markersize=10, label='EHE1 - large', color='red', linewidth=2)
ax.set_xlabel(r'$\log_{10} \langle M_b \rangle$', fontsize=20)
ax.set_ylabel('Max plume area', fontsize=20)
ax.legend(fontsize=16)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig = plt.figure(figsize=(12, 7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
ax.plot(bcs, ave_cv_ctl, marker='o', markersize=10, label='CTL - large', color='black', linewidth=2)
ax.plot(bcs, ave_cv_ehe1, marker='s', markersize=10, label='EHE1 - large', color='red', linewidth=2)
ax.set_xlabel(r'$\log_{10} \langle M_b \rangle$', fontsize=20)
ax.set_ylabel('Cloud Volume (km^3)', fontsize=20)
ax.set_yscale('log')
ax.legend(fontsize=16)
ax.grid(True, alpha=0.3)
plt.show()